<a href="https://colab.research.google.com/github/vicente-gonzalez-ruiz/medical_imaging/blob/main/notebooks/unet_n2n.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Noise2Noise denoising of gray-scale images

In [ ]:
import urllib.request
from PIL import Image
import io
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

In [ ]:
# Define the U-Net (see https://arxiv.org/pdf/1803.04189) (see https://github.com/sashrika15/noise2noise/blob/master/unet.py)

class Unet(nn.Module):
    '''
    Unet architecture for n2n.
    No batch norm, dropout
    '''

    def __init__(self, in_channels=3, out_channels=3):
            """Initializes U-Net."""

            super(Unet, self).__init__()

            self._block1 = nn.Sequential(
                nn.Conv2d(in_channels, 48, 3, stride=1, padding=1),
                nn.ReLU(inplace=True),
                nn.Conv2d(48, 48, 3, padding=1),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(2))

            self._block2 = nn.Sequential(
                nn.Conv2d(48, 48, 3, stride=1, padding=1),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(2))

            self._block3 = nn.Sequential(
                nn.Conv2d(48, 48, 3, stride=1, padding=1),
                nn.ReLU(inplace=True),
                nn.ConvTranspose2d(48, 48, 3, stride=2, padding=1, output_padding=1))

            self._block4 = nn.Sequential(
                nn.Conv2d(96, 96, 3, stride=1, padding=1),
                nn.ReLU(inplace=True),
                nn.Conv2d(96, 96, 3, stride=1, padding=1),
                nn.ReLU(inplace=True),
                nn.ConvTranspose2d(96, 96, 3, stride=2, padding=1, output_padding=1))

            self._block5 = nn.Sequential(
                nn.Conv2d(144, 96, 3, stride=1, padding=1),
                nn.ReLU(inplace=True),
                nn.Conv2d(96, 96, 3, stride=1, padding=1),
                nn.ReLU(inplace=True),
                nn.ConvTranspose2d(96, 96, 3, stride=2, padding=1, output_padding=1))

            self._block6 = nn.Sequential(
                nn.Conv2d(96 + in_channels, 64, 3, stride=1, padding=1),
                nn.ReLU(inplace=True),
                nn.Conv2d(64, 32, 3, stride=1, padding=1),
                nn.ReLU(inplace=True),
                nn.Conv2d(32, out_channels, 3, stride=1, padding=1),
                nn.LeakyReLU(0.1))


    def forward(self, x):

        #Encoder
        #print("X size = ", str(x.size()))
        pool1 = self._block1(x)
        #print(pool1.size())
        pool2 = self._block2(pool1)
        #print(pool2.size())
        pool3 = self._block2(pool2)
        #print(pool3.size())
        pool4 = self._block2(pool3)
        #print(pool4.size())
        pool5 = self._block2(pool4)
        #print(pool5.size())

        #Decoder
        upsample5 = self._block3(pool5)
        #print(upsample5.size())
        concat5 = torch.cat((upsample5, pool4), dim=1)
        #print(concat5.size())
        upsample4 = self._block4(concat5)
        #print(upsample4.size())
        concat4 = torch.cat((upsample4, pool3), dim=1)
        #print(concat4.size())
        upsample3 = self._block5(concat4)
        #print(upsample3.size())
        concat3 = torch.cat((upsample3, pool2), dim=1)
        #print(concat3.size())
        upsample2 = self._block5(concat3)
        #print(upsample2.size())
        concat2 = torch.cat((upsample2, pool1), dim=1)
        #print(concat2.size())
        upsample1 = self._block5(concat2)
        #print(upsample1.size())
        concat1 = torch.cat((upsample1, x), dim=1)
        #print(concat1.size())
        output = self._block6(concat1)
        #print(output.size())
        return output

    def summary(self):
        print('Unet summary: ')
        print(self._block1)
        print(self._block2)
        print(self._block3)
        print(self._block4)
        print(self._block5)
        print(self._block6)

Unet(in_channels=1, out_channels=1).summary()

In [ ]:
# Determine the "device"

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

In [ ]:
# Configure the denoiser's training

denoiser = Unet(in_channels=1, out_channels=1).to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(denoiser.parameters(), lr=0.001)
epochs = 10

In [ ]:
# Load Lena

url = "https://www.hpca.ual.es/~vruiz/images/lena_512x512_gray.png"
with urllib.request.urlopen(url) as response:
    img_data = response.read()
    clean = Image.open(io.BytesIO(img_data)).convert('L')
clean = np.array(clean, dtype=np.uint8)

In [ ]:
# Add noise

mean = 0
sigma = 25

# Ensure clean is treated as a numpy array for numpy-based noise creation
clean_np = clean.numpy() if isinstance(clean, torch.Tensor) else clean
gaussian_noise = np.random.normal(mean, sigma, clean_np.shape)
noisy = np.clip(clean_np + gaussian_noise, 0, 255).astype(np.uint8)

# Ensure clean_base is a PyTorch float tensor scaled to [0, 1]
clean_tensor = torch.from_numpy(clean_np) if isinstance(clean_np, np.ndarray) else clean
clean_base = clean_tensor.float() / 255.0

# Generate first independent noisy realization (noisy_source)
noise1 = torch.randn_like(clean_base) * (25.0 / 255.0)
noisy_source = torch.clamp(clean_base + noise1, 0.0, 1.0).unsqueeze(0).unsqueeze(0)

# Generate second independent noisy realization (noisy_target)
noise2 = torch.randn_like(clean_base) * (25.0 / 255.0)
noisy_target = torch.clamp(clean_base + noise2, 0.0, 1.0).unsqueeze(0).unsqueeze(0)

# 4. Create PyTorch Dataset and DataLoader
dataset = TensorDataset(noisy_source, noisy_target)
lena_loader = DataLoader(dataset, batch_size=1, shuffle=True)

# 5. Map the models and metrics to align with variables in the next cell's training loop
model_lena = denoiser
losses = []

# Prepare fixed noisy input scaled to [0, 1] for visual tracking
fixed_noisy_input = noisy_source.to(device)

In [ ]:
import torch
from torch.utils.data import TensorDataset, DataLoader

# Define the tile size
tile_size = 32
img_size = 512

# Ensure noisy_source and noisy_target are available (each has shape [1, 1, 512, 512])
# We will extract 32x32 tiles
tiles_source = []
tiles_target = []

for i in range(0, img_size, tile_size):
    for j in range(0, img_size, tile_size):
        # Slice out 32x32 patches
        tile_src = noisy_source[:, :, i:i+tile_size, j:j+tile_size]
        tile_tgt = noisy_target[:, :, i:i+tile_size, j:j+tile_size]

        tiles_source.append(tile_src)
        tiles_target.append(tile_tgt)

# Concatenate all tiles along the batch dimension
tiles_source_tensor = torch.cat(tiles_source, dim=0) # Shape: (256, 1, 32, 32)
tiles_target_tensor = torch.cat(tiles_target, dim=0) # Shape: (256, 1, 32, 32)

print(f"Created {tiles_source_tensor.shape[0]} tiles of shape {tiles_source_tensor.shape[2:]}")

# Create a new DataLoader for training on tiles
tile_dataset = TensorDataset(tiles_source_tensor, tiles_target_tensor)
tile_loader = DataLoader(tile_dataset, batch_size=32, shuffle=True)


In [ ]:
# Noise2Noise Training Loop using 32x32 tiles

# Re-initialize the model to train from scratch on tiles
denoiser_tiles = Unet(in_channels=1, out_channels=1).to(device)
criterion_tiles = nn.MSELoss()
optimizer_tiles = optim.Adam(denoiser_tiles.parameters(), lr=0.001)

epochs_tiles = 10
tile_losses = []

for epoch in range(1, epochs_tiles + 1):
    denoiser_tiles.train()
    epoch_loss = 0.0

    for batch_source, batch_target in tile_loader:
        batch_source = batch_source.to(device)
        batch_target = batch_target.to(device)

        optimizer_tiles.zero_grad()
        output = denoiser_tiles(batch_source)
        loss = criterion_tiles(output, batch_target)
        loss.backward()
        optimizer_tiles.step()

        epoch_loss += loss.item() * batch_source.size(0)

    epoch_loss /= len(tile_loader.dataset)
    tile_losses.append(epoch_loss)

    # Evaluate progress on the full 512x512 noisy input
    denoiser_tiles.eval()
    with torch.no_grad():
        # Feeding the full image to the model to see reconstruction progress
        current_output = denoiser_tiles(fixed_noisy_input).cpu()
    current_img = torch.clamp(current_output.squeeze(), 0.0, 1.0).numpy()

    # Display current reconstruction
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(current_img, cmap='gray')
    ax.set_title(f'Tile Training - Epoch {epoch} (Avg Loss: {epoch_loss:.6f})')
    ax.axis('off')
    plt.show()


In [ ]:
import os

# Switch model to evaluation mode
denoiser_tiles.eval()

with torch.no_grad():
    # Run model on the full 512x512 noisy input
    output_tensor = denoiser_tiles(fixed_noisy_input).cpu()

# Squeeze and convert to numpy array in [0, 255] scale
denoised_np = torch.clamp(output_tensor.squeeze(), 0.0, 1.0).numpy()
denoised_image_scaled = (denoised_np * 255.0).astype(np.uint8)

# Save image to disk
output_path = "denoised_lena.png"
saved_image = Image.fromarray(denoised_image_scaled)
saved_image.save(output_path)

print(f"Successfully saved the denoised image to: {os.path.abspath(output_path)}")
